# Accented speech, full text: segmental changes only (no prosody transfer)

1. Text → IPA, one chunk per sentence
2. `manipulate()` applies your accent rules to the IPA
3. `synth_aligned()` synthesizes native and accented versions with the **same voice and its own prosody**
4. Both versions are shown in the interactive player (altered phonemes highlighted) and saved as wav + html in `output_online/`

In [2]:
from functions import *
import os
import soundfile as sf
from IPython.display import display, HTML

c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
c:\Users\utente\anaconda3\envs\koko\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [3]:
from functions import *          # model, g2p, _encode, _voice_pack, _durations_cached, align_phonemes,
                                 # text_to_ipa_chunks, audiovisualize_interactive, manipulate, SR, FRAME_SAMPLES ...
import functions as F            # to reuse the player's HTML template
from functions import _encode, _voice_pack, _durations_cached   # underscore names are not exported by *
import torch, numpy as np, json, os, uuid
import soundfile as sf
from functions import * 
from functions import  _voice_pack, _encode,   predict_durations   , align_phonemes   # model, g2p, _encode, _voice_pack, predict_durations, align_phonemes, SR, FRAME_SAMPLES
import torch, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Audio
from IPython.display import display, HTML, Audio

HOP   = FRAME_SAMPLES // 2       # F0 / energy frame: 300 samples = 12.5 ms
HOP_S = HOP / SR

## Settings

In [6]:
text = (""" Elena Marchetti lavorava da sei anni nell'archivio comunale di Ponteverde, una piccola città costruita lungo un fiume nella pianura del Nord. L'archivio occupava il terzo piano di un vecchio palazzo di mattoni rossi, a pochi passi dalla piazza principale e dal municipio. Il palazzo era stato costruito alla fine dell'Ottocento, e in passato aveva ospitato una scuola e poi alcuni uffici del tribunale. La sala più grande aveva soffitti alti, scaffali di legno scuro e due lunghe finestre che guardavano verso il fiume. In quella sala si trovavano i registri delle nascite, dei matrimoni e delle proprietà, alcuni dei quali avevano più di duecento anni. Il lavoro di Elena consisteva nel catalogare i documenti, nel rispondere alle richieste dei cittadini e nel tenere in ordine le sale. Ogni mattina arrivava alle otto, accendeva le luci, apriva le finestre per qualche minuto e controllava la posta elettronica dell'ufficio. Di solito restava in archivio fino alle sette di sera, perché preferiva lavorare quando il palazzo era tranquillo e silenzioso. Nelle ore del pomeriggio gli altri uffici si svuotavano, e lei poteva concentrarsi sui registri più vecchi senza essere interrotta.
Con lei lavorava anche Paolo Ferretti, un collega più giovane che si occupava soprattutto della digitalizzazione dei documenti. Paolo passava molte ore davanti allo scanner, una grande macchina che fotografava le pagine una per una con molta precisione. Quando aveva tempo, Paolo aiutava Elena anche con le richieste dei cittadini, soprattutto quelle che arrivavano per posta elettronica. Al piano terra, invece, c'era il custode del palazzo, Aldo Rinaldi, che lavorava in quell'edificio da più di trent'anni. Aldo conosceva ogni stanza, ogni porta e ogni scala, e spesso raccontava come era cambiato il palazzo nel corso del tempo. Ogni giovedì Elena passava qualche minuto con Aldo nell'ingresso, per parlare del tempo, dei lavori in città e delle novità del comune.
All'inizio di novembre, una sera, Elena ha sentito per la prima volta un rumore strano che veniva dal soffitto della sala grande. Era una serie di piccoli colpi secchi, simili al suono di qualcuno che batte lentamente con le nocche su una porta di legno. Non assomigliava ai rumori normali di un edificio vecchio, come il vento nelle finestre o i passi delle persone nelle scale. Elena si è alzata dalla scrivania, ha camminato fino al centro della sala e ha guardato il soffitto per qualche momento. Ha pensato per un attimo a una finestra aperta, ma tutte le finestre della sala erano chiuse e il vento non era forte. Non ha visto niente di strano, perché l'intonaco era bianco e liscio come sempre, e le lampade erano ferme al loro posto. Il rumore è durato circa dieci minuti, poi si è fermato all'improvviso, e la sala è tornata silenziosa come sempre. Elena ha pensato che qualcuno stesse lavorando al piano di sopra, ma poi si è ricordata che il terzo piano era l'ultimo. Sopra l'archivio c'era soltanto il tetto, con le travi di legno e uno spazio basso dove nessuno entrava da molti anni.
La sera seguente, verso la stessa ora, il rumore è ricominciato, sempre dallo stesso punto del soffitto, vicino alla finestra. Anche questa volta è durato pochi minuti, e poi è scomparso senza lasciare nessuna traccia e nessuna spiegazione evidente. Elena ha preso un foglio e ha scritto la data, l'ora e una breve descrizione del suono, per non dimenticare i dettagli. Nei giorni successivi il rumore si è ripetuto quasi ogni sera, e qualche volta Elena lo ha sentito anche al mattino presto. Non era un suono forte, ma in una sala così silenziosa era impossibile non notarlo e non chiedersi da dove venisse. Per abitudine, Elena ha messo il foglio nel primo cassetto della scrivania, insieme alle chiavi e agli altri appunti della settimana.
Una mattina Elena ha raccontato tutto a Paolo, mentre preparavano insieme le scatole di documenti da portare allo scanner. Paolo l'ha ascoltata con attenzione e poi ha detto che, secondo lui, nel soffitto potevano esserci dei topi o altri piccoli animali. Ha spiegato che negli edifici vecchi succede spesso, soprattutto in autunno, quando gli animali cercano un posto caldo per l'inverno. Ha raccontato che anche nella casa dei suoi nonni, in campagna, una volta avevano trovato dei topi sotto il tetto. In quel caso, ha detto, gli animali facevano rumore soprattutto di notte, quando la casa era buia e tranquilla. "Se ci sono dei topi sopra la sala, dobbiamo chiamare subito qualcuno, perché potrebbero rovinare anche i documenti", ha detto Paolo. Elena ha pensato che l'idea fosse ragionevole, perché i colpi erano irregolari e potevano sembrare i movimenti di un animale. Tuttavia, le sembrava strano che gli animali facessero rumore sempre alla stessa ora, e quasi sempre per lo stesso numero di minuti. Inoltre, durante il giorno la sala era piena di rumori normali, e forse gli animali si muovevano anche in quelle ore senza farsi sentire. Ha deciso comunque di seguire il consiglio del collega, perché la protezione dei documenti era la cosa più importante per l'archivio. Nel pomeriggio ha scritto una richiesta all'ufficio tecnico del comune, chiedendo un controllo del soffitto e dello spazio sotto il tetto.
Pochi giorni dopo è arrivato un tecnico di una ditta specializzata, con una scala, una torcia potente e una piccola telecamera. La ditta lavorava spesso per il comune, e controllava regolarmente le scuole, le palestre e gli altri edifici pubblici della città. Il tecnico è salito nello spazio sotto il tetto attraverso una botola nel corridoio, che Elena non aveva mai notato prima. Elena e Paolo hanno aspettato nel corridoio, ascoltando i passi del tecnico che si muoveva lentamente sopra le loro teste. Ogni tanto sentivano anche la sua voce, perché parlava al telefono con un collega per descrivere quello che vedeva. Ha passato quasi un'ora a controllare le travi, gli angoli e il pavimento di legno sopra la sala grande dell'archivio. Ha usato la telecamera per guardare anche nei punti più stretti, dove una persona non riusciva a entrare con il corpo. Le immagini apparivano su un piccolo schermo, e il tecnico le osservava con calma prima di passare al punto successivo. Quando è sceso, ha spiegato a Elena e a Paolo che non aveva trovato nessun segno della presenza di animali. Non c'erano nidi, non c'erano escrementi e non c'erano buchi nel legno o nei muri, nemmeno vicino alla finestra. Ha aggiunto che lo spazio era molto pulito e asciutto, e che il tetto sembrava in buone condizioni in ogni punto. Prima di andare via ha lasciato alcune trappole per sicurezza, ma ha detto che probabilmente sarebbero rimaste vuote. Infatti, nelle due settimane seguenti le trappole sono rimaste vuote, mentre il rumore continuava a ripetersi quasi tutte le sere.
Un venerdì sera, mentre usciva dal palazzo, Elena si è fermata a parlare con Aldo, che stava chiudendo la porta principale. Gli ha raccontato la storia del rumore, del tecnico e delle trappole vuote, e gli ha chiesto se avesse mai sentito qualcosa di simile. Aldo ha ascoltato tutta la storia senza interrompere, come faceva sempre quando qualcuno gli parlava di un problema del palazzo. Poi ha detto che in trent'anni non aveva mai sentito rumori strani al terzo piano, né di giorno né di sera. Ha ricordato però che, all'inizio di ottobre, il comune aveva fatto sostituire tutto l'impianto di riscaldamento del palazzo. I lavori erano durati tre settimane, e durante quel periodo l'edificio era stato pieno di operai, di attrezzi e di polvere. Gli operai avevano installato una caldaia nuova e anche dei tubi nuovi, che passavano proprio sopra il soffitto dell'ultimo piano. "Forse non è un caso: il rumore è cominciato poche settimane dopo i lavori sul riscaldamento", ha detto Aldo con calma. Ha aggiunto che, se voleva, poteva darle gli orari del nuovo impianto, perché erano programmati con un timer automatico. Aldo ha spiegato che il timer accendeva e spegneva la caldaia ogni giorno alla stessa ora, senza bisogno di nessun intervento. Elena non aveva mai pensato al riscaldamento, perché i termosifoni della sala funzionavano bene e non facevano nessun rumore. Inoltre, i tubi non si vedevano, perché erano nascosti nello spazio sotto il tetto e nei muri spessi del palazzo. Tornando a casa, però, ha riflettuto sulle parole di Aldo e ha deciso di raccogliere informazioni in modo più preciso.
Dal lunedì seguente Elena ha preparato una tabella semplice, con una colonna per il giorno, una per l'ora e una per la durata. Aveva scelto di usare una tabella perché sapeva, dal suo lavoro in archivio, che i dati scritti sono più affidabili della memoria. Ogni volta che sentiva il rumore, guardava l'orologio e scriveva il momento esatto in cui il suono cominciava e finiva. Ha chiesto anche ad Aldo una copia degli orari del riscaldamento, che erano scritti su un foglio appeso vicino alla caldaia. Il foglio indicava che l'impianto si accendeva alle sei del mattino e si spegneva alle sei e mezza di sera, dal lunedì al venerdì. Nel fine settimana, invece, il riscaldamento restava spento, perché nel palazzo non lavorava nessuno e gli uffici erano chiusi. Nelle prime sere i numeri non le dicevano molto, ma dopo alcuni giorni Elena ha cominciato a vedere una certa regolarità. Per due settimane ha scritto i dati ogni giorno, anche nelle sere in cui il rumore era molto debole e difficile da sentire. Anche Paolo l'ha aiutata, perché nei giorni in cui Elena usciva prima, restava lui in sala e annotava gli orari nella tabella. Il rumore non arrivava mai nel mezzo della giornata, quando la sala era calda e il riscaldamento funzionava senza interruzioni. Una domenica è passata in archivio per prendere un libro, e ha notato che la sala era silenziosa per tutto il pomeriggio. Alla fine di due settimane ha messo la sua tabella accanto al foglio degli orari, e ha confrontato le due serie di numeri.
Il risultato era molto chiaro: il rumore cominciava sempre pochi minuti dopo lo spegnimento del riscaldamento, verso le sei e quaranta. In tutte le sere registrate, la differenza tra lo spegnimento dell'impianto e l'inizio dei colpi era sempre di circa dieci minuti. Al mattino, invece, i colpi arrivavano poco dopo l'accensione, mentre l'acqua calda cominciava a scorrere di nuovo nei tubi. Nei fine settimana, quando l'impianto restava spento, il rumore non si sentiva mai, né al mattino né alla sera. Elena ha mostrato la tabella ad Aldo, che l'ha letta con attenzione e ha confrontato ogni ora con il foglio della caldaia. Aldo ha detto che i numeri non lasciavano dubbi, e ha telefonato subito alla ditta che aveva installato il nuovo impianto. Qualche giorno dopo è arrivato un idraulico, che è salito nello spazio sotto il tetto insieme ad Aldo, con una torcia e alcuni attrezzi. Prima di salire, l'idraulico aveva guardato la tabella di Elena, e aveva detto che quei dati gli avrebbero fatto risparmiare molto tempo. Dopo pochi minuti ha trovato il punto: un tubo di metallo appoggiato direttamente su una vecchia trave di legno, vicino alla finestra. L'idraulico ha chiesto a Elena di restare nella sala grande, e poi ha battuto piano sul tubo con un piccolo attrezzo. Elena ha sentito lo stesso suono secco delle settimane precedenti, e ha confermato che veniva esattamente da quel punto del soffitto. L'idraulico ha spiegato che il metallo si allunga un poco quando diventa caldo e si accorcia di nuovo quando si raffredda. Ogni volta che la temperatura cambiava, il tubo scivolava sulla trave a piccoli scatti, e ogni scatto produceva un colpo secco. "È un fenomeno normale, che succede spesso quando i tubi nuovi vengono montati troppo vicino al legno", ha detto l'idraulico.
Per risolvere il problema, l'idraulico ha sollevato leggermente il tubo e ha inserito tra il metallo e la trave alcuni supporti di gomma. La gomma permetteva al tubo di muoversi liberamente, senza strisciare sul legno e quindi senza produrre rumore durante i cambi di temperatura. Secondo l'idraulico, quei supporti erano economici e semplici da montare, e potevano durare molti anni senza bisogno di manutenzione. Ha controllato anche gli altri tubi sopra il terzo piano, ma erano tutti montati correttamente e lontani dalle travi. L'idraulico ha spiegato che era meglio controllare tutto subito, perché lo stesso problema poteva comparire anche in altri punti. Prima di scendere, Aldo ha scritto la data dell'intervento nel quaderno dove registrava tutte le riparazioni del palazzo. Quella sera Elena è rimasta in archivio fino alle sette, come al solito, e ha ascoltato con attenzione il silenzio della sala. Alle sei e mezza il riscaldamento si è spento, ma dal soffitto non è arrivato nessun colpo, né in quel momento né più tardi. Il giorno dopo ha raccontato a Paolo come erano andate le cose, e insieme hanno tolto le trappole che erano rimaste nel corridoio. Paolo ha osservato che era meglio così, perché i documenti più antichi erano delicati e difficili da sostituire. Elena ha conservato la tabella in una cartella dell'archivio, insieme alla documentazione dei lavori sul riscaldamento del palazzo. Pensava che un giorno qualcun altro avrebbe potuto trovare utile quella informazione, magari durante altri lavori sull'edificio. Poi è tornata ai suoi registri antichi, e le settimane successive sono passate tranquille, tra le richieste dei cittadini e i documenti da catalogare.""")


text = text[:2000]  # limit to 4096 characters for demo
voice = "im_nicola"

# accent rules, applied in order (see the docstring at the top of functions.py for all kinds)
rules = [{"kind": "substitute_prob", "old": "t", "choices": {"tʲ": 33, "θ": 33, "ɖ": 33}, "same_in_word": True},
        {"kind": "substitute_prob", "old": "k", "choices": {"x": 33, "χ": 33,"kʲ": 33 }, "same_in_word": True},
        {"kind": "substitute_prob", "old": "p", "choices": {"β": 33, "ɸ": 33, "pʲ": 33}, "same_in_word": True}
        # {"kind": "substitute_prob", "old": "e", "choices": {"ø": 100}, "same_in_word": True} 'β', 'ɸ', 'pʰ', 'pʲ'
        ]    # <- put your manipulation rules here
    # <- put your manipulation rules here


SEED        = 1      # reproducible random substitutions
SAME_TIMING = False   # True : accented version reuses the native durations -> identical word boundaries
                     # False: the voice times the accented string itself

## Text → IPA → accented IPA

In [7]:
ipa_chunks = text_to_ipa_chunks(text)

accent_chunks, masks, stats, origins = manipulate(
    ipa_chunks, rules, return_mask=True, return_stats=True, return_origin=True, seed=SEED)

for orig, acc in zip(ipa_chunks, accent_chunks):
    print("native  :", orig)
    print("accented:", acc, "\n")
print_stats(stats)

assert check_ipa(accent_chunks), "some symbols are not in Kokoro's vocab and would be dropped (see above)"

native  : ˈɛːlena markˈetːɪ lavorˈava da sˌɛi ˈannɪ nellarkˈivio komʊnˈale dɪ pontevˈɛrde, ˌʊna pˈikːola ʧitːˈa kostrʊˈita lˈuŋɡo ʊn fjˈume nˈella pjanˈura del nˈɔrd.
accented: ˈɛːlena marχˈetʲːɪ lavorˈava da sˌɛi ˈannɪ nellarχˈivio χomʊnˈale dɪ βonɖevˈɛrde, ˌʊna pʲˈikʲːola ʧiɖːˈa xostʲrʊˈitʲa lˈuŋɡo ʊn fjˈume nˈella ɸjanˈura del nˈɔrd. 

native  : larkˈivio okːʊpˈava il tˈɛrʦo pjˈano dɪ ʊn vˈɛkːio palˈaʦːo dɪ matːˈonɪ rˈoSɪ, a pˈɔkɪ pˈaSɪ dˌalla pjˈaʦːa prinʧipˈale e dal mʊniʧˈipio.
accented: larχˈivio okʲːʊpʲˈava il ɖˈɛrʦo βjˈano dɪ ʊn vˈɛxːio βalˈaʦːo dɪ matʲːˈonɪ rˈoSɪ, a ɸˈɔkʲɪ pʲˈaSɪ dˌalla ɸjˈaʦːa βrinʧiβˈale e dal mʊniʧˈiɸio. 

native  : il palˈaʦːo ˌɛra stˈato kostrʊˈito ˌalla fˈine dellotːoʧˈɛnto, e in paSˈato avˌeva ospitˈato ˌʊna skʊˈɔla e pˈɔi alkˌʊnɪ ʊffˈiʧɪ del tribʊnˈale.
accented: il βalˈaʦːo ˌɛra stʲˈatʲo xostʲrʊˈitʲo ˌalla fˈine delloθːoʧˈɛnθo, e in pʲaSˈaθo avˌeva osɸitʲˈatʲo ˌʊna sχʊˈɔla e ɸˈɔi alxˌʊnɪ ʊffˈiʧɪ del tʲribʊnˈale. 

native  : la sˈala pjʊ ɡrˈande avˌev

## Native

In [ ]:
audio_nat, segs_nat, words_nat, alt_nat = synth_aligned(ipa_chunks, voice)

display(HTML(audiovisualize_interactive(audio_nat, segs_nat, seg_words=words_nat, text=text,
                                        seg_altered=alt_nat, title=f"Native: {voice}",
                                        out_html="native.html")))

## Accented

In [8]:
if SAME_TIMING:
    audio_acc, segs_acc, words_acc, alt_acc = synth_aligned(
        accent_chunks, voice, masks=masks,
        dur_source="original", original_chunks=ipa_chunks, origins=origins)
else:
    audio_acc, segs_acc, words_acc, alt_acc = synth_aligned(
        accent_chunks, voice, masks=masks, dur_source="manipulated")

audiovisualize_interactive(audio_acc, segs_acc, seg_words=words_acc, text=text,
                                        seg_altered=alt_acc, title=f"Accented: {voice}",
                                        rules=rules, stats=stats,
                                        out_html="accented_segmentaltkp.html")

Writing interactive HTML to: c:\Users\utente\Documents\GitHub\PhD-HUM-AI-N-ACCENT\artificial_stim\output_online\accented_segmentaltkp.html


'\n<div id="pv" style="font-family:system-ui,sans-serif;max-width:1080px">\n<h2 id="pageTitle" style="margin:0 0 4px;font-size:20px;font-weight:700;color:#111;"></h2>\n<div id="rulesBox" style="font-size:12.5px;color:#555;margin-bottom:12px;padding:7px 11px;background:#f3f4f6;border-radius:6px;display:none;max-height:60px;overflow-y:auto;"></div>\n<div id="statsBox" style="display:none;margin:0 0 12px;padding:9px 12px;border:1px solid #e5e7eb;border-radius:8px;background:#fffdf7;font-size:13px;color:#333;">\n    <div style="font-weight:700;margin-bottom:6px;color:#111;">Substitutions <span id="statsTotal" style="font-weight:400;color:#666;"></span></div>\n    <div id="statsBody"></div>\n</div>\n<audio id="au" src="..."></audio>\n<div style="display:flex;gap:8px;align-items:center;margin-bottom:2px">\n    <button id="pp" style="padding:6px 14px;border:0;border-radius:6px;background:#2563eb;color:#fff;cursor:pointer">▶ play</button>\n    <input id="sp" type="range" min="0.5" max="1.5" st

## Save the audio

In [ ]:
os.makedirs("output_online", exist_ok=True)
sf.write("output_online/native.wav", audio_nat, SR)
sf.write("output_online/accented_segmental.wav", audio_acc, SR)
print(f"native   {len(audio_nat)/SR:6.2f} s")
print(f"accented {len(audio_acc)/SR:6.2f} s")